# The impact of cell segmentation on spatial transcriptomics analysis

**EBI Workshop: Data-Driven Approaches to Understanding Dementia**

This notebook explores how different cell segmentation strategies affect downstream analysis of spatial transcriptomics data from the 10x Genomics Xenium platform. Using postmortem human brain tissue, we examine how segmentation choices influence:

- The number and shape of cell masks
- Transcript assignment efficiency and per-cell metrics
- Cell type identification, clustering, and gene expression specificity

By the end of this notebook you will understand why segmentation is a critical preprocessing step and how to evaluate different approaches.

## The Dataset

The data displayed here is from **Kotah et al. (2025)**, *"Beyond the nuclear border: single-cell analysis of in situ sequenced human brain tissue using cellular features"* ([Nature Communications Biology](https://www.nature.com/articles/s42003-025-08518-6)), used with the kind permission of [Janssen M. Kotah](https://jmkotah.github.io/).

The authors generated Xenium in situ sequencing (ISS) data from **formalin-fixed paraffin-embedded (FFPE) postmortem human brain tissue** (superior temporal / parietal gyrus) using the **266-gene Human Brain Panel**. They explored how different cell segmentation methods affect transcript allocation and cell type annotation.

| Resource | Link |
|----------|------|
| **Paper** | [Nature Comms Bio](https://www.nature.com/articles/s42003-025-08518-6) |
| **Dataset** | [Zenodo (DOI: 10.5281/zenodo.15425563)](https://zenodo.org/records/15425563) |
| **Code** | [GitHub](https://github.com/jmkotah/xenium_segmentation_paper) |

### Experiments

| Sample | Tissue | Segmentation methods |
|--------|--------|---------------------|
| **XE1** | Superior temporal gyrus (control) | xeNuc, xeCell (2.5 / 5 / 10 µm expansion) |
| **XE2** | Superior parietal gyrus (control) | xeNuc, xeCell (multimodal), gs18S |
| **XE3** | Superior parietal gyrus (AD) | xeCell (multimodal) |
| **XE4** | Superior parietal gyrus (control) | xeCell (multimodal) |

We focus on **XE1** and **XE2** for segmentation comparison because they have been processed to generate a number of different segmentation outputs.


## Segmentation Methods

A critical step in spatial transcriptomics is **cell segmentation** — defining the boundaries of individual cells so that detected transcripts can be assigned to them. The choice of segmentation strategy directly affects all downstream analyses.

### Nuclear segmentation (xeNuc)
The most conservative approach: cell boundaries are the **DAPI-stained nuclear borders** identified by the Xenium instrument. This is equivalent to treating the data as spatially resolved single-nucleus transcriptomes. Very accurate for cell identity, but typically captures only ~25 % of transcripts in human brain, since most mRNA in neural cells resides in the cytoplasm.

### Expanded nuclear segmentation (xeCell)
The Xenium platform algorithmically **expands nuclear borders by a fixed distance** (default 5 µm) to approximate the full cell body. This captures more transcripts but risks assigning transcripts from neighbouring cells, especially in densely packed tissue. The expansion distance is a key parameter:

- **2.5 µm** — moderate expansion, preserves most cell-type distinctions
- **5 µm** (default) — standard expansion, good transcript capture
- **10 µm** — aggressive expansion, maximum transcript capture but highest misallocation risk

### Multimodal segmentation (xeMultimodal, XE2)
Uses the **Multi-Tissue Stain Mix kit** (cell membrane antibodies, 18S RNA probe, intracellular protein stains) to expand nuclear borders in a **morphology-guided** manner rather than by a fixed distance. This works very well in some tissues and not so well in others. Not all components of the multimodal stain work that well in brain samples unfortunately, and the search for good protein based cell boundary stains in the brain still requires advancement.


## 1. Setup and Data Loading

First, import modules - including spatialdata and other plotting modules

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import scanpy as sc
import squidpy as sq
import spatialdata as sd
from shapely.geometry import box

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sc.settings.set_figure_params(dpi=100, frameon=False)
sns.set_style("whitegrid")
plt.rcParams["figure.facecolor"] = "white"


We load pre-built `SpatialData` objects that bundle every element (images, transcripts, cell boundaries, count tables) for each experiment into a single on-disk zarr store. The raw data from this paper was uploaded to Zenodo as individual components, so a notebook for creating these spatialdata objects for this course is included at XXX if you are interested. This has been pre-ran for you to save time.

In [ ]:
DATA_ROOT = Path("/home/training/course_dir/data_dir/intro_spatialdata/datasets/segmentation_spatialdata_objects")

sdata_xe1 = sd.read_zarr(DATA_ROOT / "XE1.zarr")
sdata_xe2 = sd.read_zarr(DATA_ROOT / "XE2.zarr")

print(sdata_xe1)
print()
print(sdata_xe2)

The next cell defines everything we need to work with **XE1** in a consistent way.

**`XE1_METHODS`** is a small lookup table: each key is a human-readable segmentation name, and each value points to the matching **`AnnData` table** and **polygon boundaries** inside the `SpatialData` object (`sdata_xe1`). Those element names come from how the Kotah et al. pipeline exported the data. We use the same keys (`"xeNuc"`, `"xeCell 5um"`, etc.) as labels in plots later.

- **`xeNuc`** — transcripts and counts assigned using **nucleus-only** masks (no cytoplasmic expansion).
- **`xeCell 2.5µm` / `5µm` / `10µm`** — the same nuclear seeds expanded by a **fixed radius** in microns to approximate whole-cell territories (larger radius → more transcripts assigned per “cell”, but more risk of merging neighbours).

**`CELLTYPE_ORDER`** and **`CELLTYPE_PALETTE`** keep cell-type colours and legend order the same across figures. **`harmonize_adata`** tidies labels (e.g. `Astrocyte` → `Astro`) and builds a categorical `celltype_broad` column with matching colours in `adata.uns`.

Finally we build **`tables_xe1`**: a dictionary of in-memory `AnnData` copies (one per method) that we use for UMAP, clustering metrics, and bar charts without re-reading the zarr store each time.


In [ ]:
# Segmentation method metadata for XE1
XE1_METHODS = {
    "xeNuc": {
        "table": "XE1_xeNuc_table",
        "boundaries": "XE1_xeNuc_boundaries",
    },
    "xeCell 2.5um": {
        "table": "XE1_xeCell_2.5um_table",
        "boundaries": "XE1_xeCell_2.5um_cell_boundaries",
    },
    "xeCell 5um": {
        "table": "XE1_xeCell_5um_table",
        "boundaries": "XE1_xeCell_5um_cell_boundaries",
    },
    "xeCell 10um": {
        "table": "XE1_xeCell_10um_table",
        "boundaries": "XE1_xeCell_10um_cell_boundaries",
    },
}

# Consistent cell-type colour palette used across all plots
CELLTYPE_ORDER = [
    "ExcNeu", "InhNeu", "Oligo", "OPC", "Astro",
    "Microglia", "Endothelial", "VLMC", "CAMs",
]
CELLTYPE_PALETTE = {
    "ExcNeu": "#1f77b4",
    "InhNeu": "#ff7f0e",
    "Oligo": "#2ca02c",
    "OPC": "#d62728",
    "Astro": "#9467bd",
    "Microglia": "#8c564b",
    "Endothelial": "#e377c2",
    "VLMC": "#7f7f7f",
    "CAMs": "#bcbd22",
}


def harmonize_adata(adata):
    adata.obs["celltype_broad"] = (
        adata.obs["celltype_broad"]
        .astype(str)
        .replace({"Astrocyte": "Astro"})
    )
    present = [ct for ct in CELLTYPE_ORDER if ct in adata.obs["celltype_broad"].values]
    adata.obs["celltype_broad"] = pd.Categorical(
        adata.obs["celltype_broad"], categories=present, ordered=True
    )
    adata.uns["celltype_broad_colors"] = [CELLTYPE_PALETTE[ct] for ct in present]
    return adata


tables_xe1 = {}
for method, info in XE1_METHODS.items():
    adata = sdata_xe1[info["table"]].copy()
    adata = harmonize_adata(adata)
    tables_xe1[method] = adata

print(f"{'Method':<20s}  {'Cells':>8s}  {'Transcripts':>14s}")
print("-" * 46)
for method, adata in tables_xe1.items():
    n = adata.n_obs
    tc = int(adata.obs["total_counts"].sum())
    print(f"{method:<20s}  {n:>8,}  {tc:>14,}")


## 2. Tissue Overview

We start by visualising the full tissue section from XE1 using squidpy. The **xeCell 5 µm** segmentation (the Xenium default) is used as a reference view, these are not the cell masks themselves but the centroid of each mask, coloured by its annotated broad cell type

In [ ]:
adata_ref = tables_xe1["xeCell 5um"]

sq.pl.spatial_scatter(
    adata_ref,
    color=["celltype_broad"],
    shape=None,
    size=0.4,
    library_id="spatial",
    figsize=(10, 9),
    title="XE1 tissue section — xeCell 5 um segmentation",
)
plt.show()


Here are some example genes demonstrating different spatial expression patterns

In [ ]:
# Marker gene expression across the tissue
sq.pl.spatial_scatter(
    adata_ref,
    color=["SLC17A7", "MOG", "AIF1"],
    shape=None,
    size=0.3,
    library_id="spatial",
    cmap="Reds",
    title=["SLC17A7 (excitatory neurons)", "MOG (oligodendrocytes)", "AIF1 (microglia)"],
)
plt.show()


## 3. Zooming In: Cells, Boundaries, and Transcripts

To understand segmentation at the single-cell level we zoom into a small **500 x 500 µm** region. The left panel shows the **DAPI** image from which segmentation masks are generated; the right panel shows the raw **transcript detections** as individual points.

> **Tip:** adjust `ZOOM` below to explore different parts of the tissue.

In [ ]:
ZOOM = (3600, 3600, 4000, 4000)  # (x_min, y_min, x_max, y_max) in um
x_min, y_min, x_max, y_max = ZOOM
PIXEL_SIZE = 0.2125  # Xenium default: µm per pixel

# Crop shapes and transcripts to the zoom region
sdata_xe1_zoom = sd.bounding_box_query(
    sdata_xe1,
    axes=("x", "y"),
    min_coordinate=[x_min, y_min],
    max_coordinate=[x_max, y_max],
    target_coordinate_system="global",
)

# Crop the morphology image in pixel coordinates (direct zarr slice)
img = sdata_xe1["morphology_focus"]
px_y0, px_y1 = int(y_min / PIXEL_SIZE), int(y_max / PIXEL_SIZE)
px_x0, px_x1 = int(x_min / PIXEL_SIZE), int(x_max / PIXEL_SIZE)
img_crop = img[0, px_y0:px_y1, px_x0:px_x1].values

# High-quality gene transcripts in the zoom window
txpts_raw = sdata_xe1_zoom["transcripts"]
txpts_zoom = txpts_raw[
    (txpts_raw["is_gene"] == True) & (txpts_raw["qv"] > 20)
].compute()

# Cell count derived from the cropped cell boundaries
n_cells_zoom = len(sdata_xe1_zoom["XE1_xeCell_5um_cell_boundaries"])
print(f"Cells in zoom window:  {n_cells_zoom:,}")
print(f"Transcripts in window: {len(txpts_zoom):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Panel 1 — morphology image (DAPI)
axes[0].imshow(
    img_crop, cmap="gray_r",
    extent=[x_min, x_max, y_max, y_min],
    aspect="equal",
)
axes[0].set_title("Morphology (DAPI)")

# Panel 2 — raw transcript points
axes[1].scatter(txpts_zoom["x"], txpts_zoom["y"], c="black", s=0.1, alpha=0.3)
axes[1].set_xlim(x_min, x_max)
axes[1].set_ylim(y_max, y_min)
axes[1].set_title(f"Transcript detections ({len(txpts_zoom):,} points)")
axes[1].set_aspect("equal")

for ax in axes:
    ax.set_xlabel("x (um)")
    ax.set_ylabel("y (um)")

plt.tight_layout()
plt.show()


## 4. Comparing Segmentation Boundaries

Now we compare how the **four** segmentation methods delineate cells in **the same tissue region**. Each panel shows cell masks (green outlines) overlaid on transcript detections (grey dots).

**What to look for:**
- **xeNuc** masks are small and tight around nuclei — many transcripts fall outside any mask
- **xeCell** masks grow with increasing expansion distance; at 10 µm, neighbouring masks nearly all touch


In [ ]:
boundary_keys = {
    "xeNuc": "XE1_xeNuc_boundaries",
    "xeCell 2.5um": "XE1_xeCell_2.5um_cell_boundaries",
    "xeCell 5um": "XE1_xeCell_5um_cell_boundaries",
    "xeCell 10um": "XE1_xeCell_10um_cell_boundaries",
}

methods = list(boundary_keys.keys())
n_methods = len(methods)
fig, axes = plt.subplots(n_methods, 2, figsize=(12, n_methods * 5.5))
roi = box(x_min, y_min, x_max, y_max)

for row, method in enumerate(methods):
    bkey = boundary_keys[method]
    gdf = sdata_xe1[bkey]
    gdf_sub = gdf[gdf.intersects(roi)]

    # Left column — morphology image + boundary overlay
    ax_img = axes[row, 0]
    ax_img.imshow(
        img_crop, cmap="gray",
        extent=[x_min, x_max, y_max, y_min],
        aspect="equal",
    )
    gdf_sub.plot(ax=ax_img, edgecolor="green", facecolor="none", linewidth=0.5)
    ax_img.set_title(f"{method} — morphology + boundaries ({len(gdf_sub):,} masks)", fontsize=10)
    ax_img.set_xlim(x_min, x_max)
    ax_img.set_ylim(y_max, y_min)
    ax_img.set_xticks([])
    ax_img.set_yticks([])

    # Right column — transcripts + boundary overlay
    ax_tx = axes[row, 1]
    ax_tx.scatter(txpts_zoom["x"], txpts_zoom["y"], c="grey", s=0.05, alpha=0.5)
    gdf_sub.plot(ax=ax_tx, edgecolor="green", facecolor="none", linewidth=0.5)
    ax_tx.set_xlim(x_min, x_max)
    ax_tx.set_ylim(y_max, y_min)
    ax_tx.set_title(f"{method} — transcripts + boundaries", fontsize=10)
    ax_tx.set_aspect("equal")
    ax_tx.set_xticks([])
    ax_tx.set_yticks([])

plt.suptitle("Segmentation boundaries across methods — same tissue region", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


**Observations:**

- The **xeNuc** masks cover only the dense nuclear centres, leaving most transcript detections outside any mask.
- As the expansion distance grows from **2.5 to 5 to 10 µm**, masks enlarge uniformly. At 10 µm, neighbouring cells' masks almost all touch, increasing the chance of transcript misallocation.


## 5. Geometric Properties of Cell Masks

Before examining downstream effects, let's characterise the masks themselves: how many are there, and how large are they?

In [ ]:
rows = []
for method, info in XE1_METHODS.items():
    gdf = sdata_xe1[info["boundaries"]]
    areas = gdf.geometry.area
    rows.append({
        "Method": method,
        "Total masks": f"{len(gdf):,}",
        "Mean area (um2)": f"{areas.mean():.1f}",
        "Median area (um2)": f"{areas.median():.1f}",
        "Std area (um2)": f"{areas.std():.1f}",
        "Total coverage (mm2)": f"{areas.sum() / 1e6:.2f}",
    })

summary_df = pd.DataFrame(rows)
summary_df

# Distribution of mask areas (subsampled to 10 000 per method for plotting speed)
area_frames = []
np.random.seed(42)
for method, info in XE1_METHODS.items():
    gdf = sdata_xe1[info["boundaries"]]
    areas = gdf.geometry.area.values
    n = min(len(areas), 10_000)
    idx = np.random.choice(len(areas), n, replace=False)
    area_frames.append(pd.DataFrame({"Method": method, "Cell area (um2)": areas[idx]}))

area_df = pd.concat(area_frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(
    data=area_df, x="Method", y="Cell area (um2)",
    cut=0, inner="box", ax=ax, palette="Set2",
)
ax.set_title("Distribution of cell mask areas by segmentation method")
ax.set_ylim(0, 800)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 6. Transcript Assignment

The primary purpose of cell segmentation is to assign detected transcripts to cells. Larger masks capture more transcripts — but at the risk of assigning transcripts from neighbouring cells. Let's quantify this trade-off.

In [ ]:
# Count total high-quality gene transcripts (this may take ~30 seconds)
print("Counting gene transcripts with QV > 20 ...")
txpts_all = sdata_xe1["transcripts"]
total_gene_txpts = int(
    txpts_all[(txpts_all["is_gene"] == True) & (txpts_all["qv"] > 20)]
    .shape[0]
    .compute()
)
print(f"Total gene transcripts (QV > 20): {total_gene_txpts:,}")
print()

# Assignment rates per segmentation method
assignment_data = []
for method, adata in tables_xe1.items():
    assigned = int(adata.obs["total_counts"].sum())
    pct = 100 * assigned / total_gene_txpts
    assignment_data.append({
        "Method": method,
        "Cells": adata.n_obs,
        "Assigned transcripts": assigned,
        "% assigned": pct,
    })

assign_df = pd.DataFrame(assignment_data)
print(assign_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    assign_df["Method"], assign_df["% assigned"],
    color=sns.color_palette("viridis", len(assign_df)),
)
ax.set_ylabel("Transcripts assigned to cells (%)")
ax.set_title("Transcript assignment efficiency by segmentation method")
for bar, pct in zip(bars, assign_df["% assigned"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f"{pct:.1f}%",
        ha="center", va="bottom", fontsize=10,
    )
ax.set_ylim(0, 100)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Per-cell QC metrics (subsampled to 5000 per method)
qc_frames = []
np.random.seed(42)
for method, adata in tables_xe1.items():
    df = adata.obs[["total_counts", "n_genes_by_counts"]].copy()
    df["Method"] = method
    if len(df) > 5000:
        df = df.sample(5000, random_state=42)
    qc_frames.append(df)

qc_df = pd.concat(qc_frames, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.violinplot(
    data=qc_df, x="Method", y="total_counts",
    cut=0, inner="box", ax=axes[0], palette="Set2",
)
axes[0].set_title("Total transcripts per cell")
axes[0].set_ylabel("Transcripts / cell")
axes[0].tick_params(axis="x", rotation=20)

sns.violinplot(
    data=qc_df, x="Method", y="n_genes_by_counts",
    cut=0, inner="box", ax=axes[1], palette="Set2",
)
axes[1].set_title("Unique genes detected per cell")
axes[1].set_ylabel("Genes / cell")
axes[1].tick_params(axis="x", rotation=20)

plt.suptitle("Per-cell quality metrics across segmentation methods", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Cell Type Identification Across Segmentation Methods

Each segmentation variant was independently clustered and annotated. All methods identify the same major CNS cell types, but with important differences in proportions — particularly for cell types that are vulnerable to transcript misallocation.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (method, adata) in zip(axes, tables_xe1.items()):
    sc.pl.umap(
        adata, color="celltype_broad", ax=ax, show=False,
        title=method, legend_loc="none", size=5,
    )

handles = [
    mpatches.Patch(color=CELLTYPE_PALETTE[ct], label=ct)
    for ct in CELLTYPE_ORDER
    if any(ct in tables_xe1[m].obs["celltype_broad"].cat.categories for m in tables_xe1)
]
fig.legend(
    handles=handles, loc="lower center", ncol=len(handles),
    fontsize=18, bbox_to_anchor=(0.5, -0.06),
)
plt.suptitle("UMAP embeddings coloured by cell type", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Cell type proportions — stacked bar chart
method_order = list(XE1_METHODS.keys())

prop_data = []
for method, adata in tables_xe1.items():
    counts = adata.obs["celltype_broad"].value_counts(normalize=True)
    for ct in CELLTYPE_ORDER:
        prop_data.append({
            "Method": method,
            "Cell Type": ct,
            "Proportion": counts.get(ct, 0),
        })

prop_df = pd.DataFrame(prop_data)
pivot = prop_df.pivot(index="Method", columns="Cell Type", values="Proportion")
pivot = pivot[[ct for ct in CELLTYPE_ORDER if ct in pivot.columns]]
pivot = pivot.reindex(method_order)

fig, ax = plt.subplots(figsize=(10, 6))
pivot.plot(
    kind="bar", stacked=True, ax=ax,
    color=[CELLTYPE_PALETTE[ct] for ct in pivot.columns],
    width=0.7,
)
ax.set_ylabel("Proportion of cells")
ax.set_title("Cell type proportions across segmentation methods")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### The VLMC problem — a case study in misallocation

**Vascular leptomeningeal cells (VLMCs)** are in close anatomical proximity to **endothelial cells** at the neurovascular unit. Notice that:

- With **xeNuc** (nuclear-only), VLMCs form a distinct cluster (~2,000 cells)
- At **2.5 µm expansion**, VLMCs are still detected (~2,900 cells)
- At **5 µm** and **10 µm** expansion, the VLMC cluster **disappears entirely**

What happened? As nuclear masks expand, endothelial transcripts "leak" into VLMC nuclei (and vice versa). The VLMC transcriptional signal is diluted and these cells are reclassified as endothelial. This demonstrates how uniform nuclear expansion can **mask real biological heterogeneity**.

In [ ]:
# Quantify VLMC, microglia, and endothelial counts across methods
ct_focus = ["VLMC", "Microglia", "Endothelial"]
focus_data = []
for method, adata in tables_xe1.items():
    for ct in ct_focus:
        if ct in adata.obs["celltype_broad"].cat.categories:
            n = int((adata.obs["celltype_broad"] == ct).sum())
        else:
            n = 0
        focus_data.append({"Method": method, "Cell Type": ct, "Count": n})

focus_df = pd.DataFrame(focus_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, ct in zip(axes, ct_focus):
    sub = focus_df[focus_df["Cell Type"] == ct]
    bars = ax.bar(sub["Method"], sub["Count"], color=CELLTYPE_PALETTE.get(ct, "grey"))
    ax.set_title(ct, fontsize=12)
    ax.set_ylabel("Number of cells")
    ax.tick_params(axis="x", rotation=20)
    for i, (_, row) in enumerate(sub.iterrows()):
        ax.text(i, row["Count"] + 150, f'{row["Count"]:,}', ha="center", fontsize=9)
    ax.grid(False)
plt.suptitle("Key cell types across segmentation methods", fontsize=13)
plt.tight_layout()

plt.show()

## 8. Marker Gene Specificity — Nuclear vs Expanded Segmentation

A key consequence of expanding nuclear segmentation boundaries is **contamination**: transcripts from neighbouring cells are absorbed, diluting cell-type-specific expression signatures. We can visualise this directly with a **dotplot** that shows a few well-known marker genes per cell type.

Compare **xeNuc** (nuclear-only, most conservative) with **xeCell 10 µm** (largest expansion). In the expanded segmentation, dot sizes increase (more cells express each gene) while colour intensity homogenises — a hallmark of transcript misallocation across cell boundaries.


In [ ]:
marker_genes = {
    "ExcNeu":       ["SLC17A7", "CRYM"],
    "InhNeu":       ["GAD1", "GAD2"],
    "Oligo":        ["MOBP", "MOG"],
    "OPC":          ["PDGFRA", "VCAN"],
    "Astro":        ["AQP4", "GJA1"],
    "Microglia":    ["AIF1", "CX3CR1"],
    "Endothelial":  ["FLT1", "PECAM1"],
    "VLMC":         ["DCN", "FBLN1"],
}

groupby_order = [ct for ct in CELLTYPE_ORDER if ct in marker_genes]
gene_list = [g for ct in groupby_order for g in marker_genes[ct]]

comparisons = [
    ("xeNuc",       "xeNuc (nuclear only)"),
    ("xeCell 10um", "xeCell 10 µm expansion"),
]

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

for idx, (ax, (method, label)) in enumerate(zip(axes, comparisons)):
    adata = tables_xe1[method].copy()
    missing = [ct for ct in groupby_order if ct not in adata.obs["celltype_broad"].values]
    if missing:
        import anndata as ad
        import scipy.sparse as sp
        dummy = ad.AnnData(
            X=sp.csr_matrix((len(missing), adata.n_vars)),
            obs=pd.DataFrame({"celltype_broad": missing}, index=[f"_dummy_{ct}" for ct in missing]),
            var=adata.var,
        )
        adata = ad.concat([adata, dummy], join="outer")
        adata.obs["celltype_broad"] = pd.Categorical(
            adata.obs["celltype_broad"], categories=groupby_order, ordered=True
        )
    dp_axes = sc.pl.dotplot(
        adata,
        var_names=gene_list,
        groupby="celltype_broad",
        categories_order=groupby_order,
        standard_scale="var",
        ax=ax,
        show=False,
    )
    main_ax = dp_axes["mainplot_ax"]

    if idx == 0:
        main_ax.tick_params(bottom=False, labelbottom=False, top=True, labeltop=True)
        for lbl in main_ax.get_xticklabels():
            lbl.set_rotation(90)
            lbl.set_ha("center")
            lbl.set_va("bottom")
    else:
        main_ax.tick_params(bottom=True, labelbottom=True, top=False, labeltop=False)

    size_legend_ax = dp_axes.get("size_legend_ax")
    if size_legend_ax is not None:
        size_legend_ax.set_title(label, fontsize=11, fontweight="bold")

plt.subplots_adjust(hspace=0.01)
plt.show()


In [ ]:
# DEG specificity: mean absolute log-fold change of top markers per cell type
def get_mean_top_lfc(adata, n_top=5):
    rgg = adata.uns["rank_genes_groups"]
    lfc = rgg["logfoldchanges"]
    if isinstance(lfc, dict):
        lfc_df = pd.DataFrame(lfc)
    elif hasattr(lfc, "dtype") and lfc.dtype.names:
        lfc_df = pd.DataFrame({n: lfc[n] for n in lfc.dtype.names})
    else:
        lfc_df = pd.DataFrame(lfc)
    return lfc_df.iloc[:n_top].abs().mean().mean()


deg_data = []
for method, adata in tables_xe1.items():
    try:
        mean_lfc = get_mean_top_lfc(adata)
        deg_data.append({"Method": method, "Mean |logFC| (top 5)": mean_lfc})
    except Exception as e:
        print(f"  {method}: skipped - {e}")

deg_df = pd.DataFrame(deg_data)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    deg_df["Method"], deg_df["Mean |logFC| (top 5)"],
    color=sns.color_palette("mako", len(deg_df)),
)
ax.set_ylabel("Mean |log fold change|")
ax.set_title("DEG specificity — mean |logFC| of top 5 markers per cell type")
for bar, v in zip(bars, deg_df["Mean |logFC| (top 5)"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
        f"{v:.2f}", ha="center", fontsize=10,
    )
plt.grid(False)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## XE2: Multimodal Cell Segmentation with the Multi-Tissue Stain Mix

While XE1 relied solely on **DAPI nuclear staining** — requiring a fixed-radius expansion
to approximate cell boundaries — experiment XE2 used the
**10x Genomics Multi-Tissue Stain Mix kit**, which provides four imaging channels:

| Channel | Stain target | Biological role |
|---------|-------------|-----------------|
| 0 — DAPI | DNA | Nuclear marker (present in all cells) |
| 1 — ATP1A1 / CD45 / E-Cadherin | Membrane proteins | **Cell boundary markers** — outlines the plasma membrane |
| 2 — 18S rRNA | Ribosomal RNA | Highlights **RNA-rich cytoplasm** (strongest in neurons) |
| 3 — αSMA / Vimentin | Cytoskeletal proteins | Marks **mesenchymal / stromal cells** (e.g. VLMC, pericytes) |

With membrane-level information, the Xenium cell segmentation algorithm
(**xeMultimodal**) can draw boundaries that follow the actual cell membrane rather
than expanding a fixed distance from the nucleus.

Below we compare two approaches on the same tissue:

1. **xeNuc** — nucleus-only segmentation (DAPI only, no expansion)
2. **xeMultimodal** — multimodal segmentation using all four stain channels

Note - the cell below will take a few minutes to run as we recompute the UMAP embedding for the xeNuc and xeMultimodal assigned transcripts


In [ ]:
XE2_METHODS = {
    "xeNuc": {
        "table": "XE2_xeNuc_table",
        "boundaries": "XE2_xeNuc_boundaries",
    },
    "xeMultimodal": {
        "table": "XE2_xeCell_table",
        "boundaries": "XE2_xeCell_cell_boundaries",
    },
}

tables_xe2 = {}
for method, info in XE2_METHODS.items():
    adata = sdata_xe2[info["table"]].copy()
    adata = harmonize_adata(adata)
    sc.pp.pca(adata, n_comps=30)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)
    tables_xe2[method] = adata

print(f"{'Method':<18s}  {'Cells':>8s}  {'Transcripts':>14s}")
print("-" * 44)
for method, adata in tables_xe2.items():
    n = adata.n_obs
    tc = int(adata.obs["total_counts"].sum())
    print(f"{method:<18s}  {n:>8,}  {tc:>14,}")


In [ ]:
ZOOM_XE2 = (4000, 1500, 4400, 1900)
x2_min, y2_min, x2_max, y2_max = ZOOM_XE2

# Crop shapes and transcripts to the zoom region
sdata_xe2_zoom = sd.bounding_box_query(
    sdata_xe2,
    axes=("x", "y"),
    min_coordinate=[x2_min, y2_min],
    max_coordinate=[x2_max, y2_max],
    target_coordinate_system="global",
)

# Crop the four image channels in pixel coordinates (direct zarr slice)
img_xe2 = sdata_xe2["morphology_focus"]
px_y0 = int(y2_min / PIXEL_SIZE)
px_y1 = int(y2_max / PIXEL_SIZE)
px_x0 = int(x2_min / PIXEL_SIZE)
px_x1 = int(x2_max / PIXEL_SIZE)

ch_dapi     = img_xe2[0, px_y0:px_y1, px_x0:px_x1].values.astype(np.float32)
ch_boundary = img_xe2[1, px_y0:px_y1, px_x0:px_x1].values.astype(np.float32)
ch_18s      = img_xe2[2, px_y0:px_y1, px_x0:px_x1].values.astype(np.float32)
ch_interior = img_xe2[3, px_y0:px_y1, px_x0:px_x1].values.astype(np.float32)

def _norm(arr, lo_pct=1, hi_pct=99.5):
    lo, hi = np.percentile(arr, [lo_pct, hi_pct])
    return np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)

rgb_xe2 = np.stack([
    _norm(ch_interior),
    _norm(ch_18s),
    #_norm(0.5 * ch_boundary + 0.5 * ch_interior),
    _norm(ch_dapi),
], axis=-1)

dapi_crop_xe2 = _norm(ch_dapi)
morph_maxproj_xe2 = np.maximum.reduce([
    _norm(ch_dapi),
    _norm(ch_18s),
    _norm(ch_interior),
])

# High-quality gene transcripts in the zoom region
gene_panel_xe2 = set(tables_xe2["xeMultimodal"].var_names)
txpts_xe2_raw = sdata_xe2_zoom["transcripts"]
txpts_xe2_zoom = txpts_xe2_raw[txpts_xe2_raw["qv"] > 20].compute()
txpts_xe2_zoom = txpts_xe2_zoom[txpts_xe2_zoom["feature_name"].isin(gene_panel_xe2)]

# Cell boundaries in the zoom region (already cropped by bounding_box_query)
gdf_nuc_zoom   = sdata_xe2_zoom["XE2_xeNuc_boundaries"]
gdf_multi_zoom = sdata_xe2_zoom["XE2_xeCell_cell_boundaries"]
# Keep roi_xe2 for use in the comparison figure below
roi_xe2 = box(x2_min, y2_min, x2_max, y2_max)

extent_xe2 = [x2_min, x2_max, y2_max, y2_min]

fig, axes = plt.subplots(3, 1, figsize=(6,17), dpi=200)

# Panel 1 — RGB composite
axes[0].imshow(rgb_xe2, extent=extent_xe2, aspect="equal")
axes[0].set_title("Multimodal staining\nR=DAPI  G=18S  B=Membrane+Interior", fontsize=10)
axes[0].grid(False)

# Panel 2 — DAPI with dual boundary overlay
axes[1].imshow(dapi_crop_xe2, cmap="gray", extent=extent_xe2, aspect="equal")
gdf_nuc_zoom.plot(ax=axes[1], edgecolor="magenta", facecolor="none", linewidth=0.6, label="xeNuc")
gdf_multi_zoom.plot(ax=axes[1], edgecolor="lime", facecolor="none", linewidth=0.6, label="xeMultimodal")
axes[1].legend(fontsize=8, loc="upper right")
axes[1].set_title(f"Boundary comparison\nmagenta = xeNuc ({len(gdf_nuc_zoom):,})  "
                   f"green = xeMultimodal ({len(gdf_multi_zoom):,})", fontsize=10)
axes[1].set_xlim(x2_min, x2_max)
axes[1].set_ylim(y2_max, y2_min)

# Panel 3 — transcripts with dual boundary overlay
axes[2].scatter(txpts_xe2_zoom["x"], txpts_xe2_zoom["y"], c="grey", s=0.05, alpha=0.6)
gdf_nuc_zoom.plot(ax=axes[2], edgecolor="magenta", facecolor="none", linewidth=0.5)
gdf_multi_zoom.plot(ax=axes[2], edgecolor="lime", facecolor="none", linewidth=0.5)
axes[2].set_xlim(x2_min, x2_max)
axes[2].set_ylim(y2_max, y2_min)
axes[2].set_title(f"Transcripts + boundaries ({len(txpts_xe2_zoom):,} points)", fontsize=10)
axes[2].set_aspect("equal")

for ax in axes:
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")

plt.suptitle("XE2 — Multimodal vs Nuclear Segmentation", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


The RGB composite reveals how each stain channel highlights different tissue structures.
DAPI (blue) marks all nuclei; 18S rRNA (green) covers the RNA-rich cytoplasm of neurons;
and the interior markers (blue) outline mesenchymal cells.

In the segmentation overlay you can clearly see how **xeMultimodal** (green contours)
extends beyond the nucleus to follow the actual cell membrane, while **xeNuc** (magenta)
captures only the nuclear footprint. This difference is especially visible around large
neurons whose cytoplasm is much larger than their nucleus.


In [ ]:
xe2_boundary_keys = {
    "xeNuc": "XE2_xeNuc_boundaries",
    "xeMultimodal": "XE2_xeCell_cell_boundaries",
}

xe2_methods_list = list(xe2_boundary_keys.keys())
n_xe2 = len(xe2_methods_list)
fig, axes = plt.subplots(n_xe2, 2, figsize=(12, n_xe2 * 5.5))

for row, method in enumerate(xe2_methods_list):
    bkey = xe2_boundary_keys[method]
    gdf = sdata_xe2[bkey]
    gdf_sub = gdf[gdf.intersects(roi_xe2)]

    # Left — top row: DAPI only; bottom row: max-projection (DAPI, 18S, interior)
    ax_img = axes[row, 0]
    img_left = dapi_crop_xe2 if row == 0 else morph_maxproj_xe2
    ax_img.imshow(img_left, cmap="gray_r", extent=extent_xe2, aspect="equal")
    gdf_sub.plot(ax=ax_img, edgecolor="green", facecolor="none", linewidth=0.5)
    morph_label = "DAPI" if row == 0 else "max proj (DAPI, 18S, interior)"
    ax_img.set_title(f"{method} — {morph_label} + boundaries ({len(gdf_sub):,} masks)", fontsize=10)
    ax_img.set_xticks([])
    ax_img.set_yticks([])
    ax_img.set_xlim(x2_min, x2_max)
    ax_img.set_ylim(y2_max, y2_min)

    # Right — transcripts + boundary overlay
    ax_tx = axes[row, 1]
    ax_tx.scatter(txpts_xe2_zoom["x"], txpts_xe2_zoom["y"], c="gray", s=0.05, alpha=0.7)
    gdf_sub.plot(ax=ax_tx, edgecolor="green", facecolor="none", linewidth=0.5)
    ax_tx.set_xlim(x2_min, x2_max)
    ax_tx.set_ylim(y2_max, y2_min)
    ax_tx.set_title(f"{method} — transcripts + boundaries", fontsize=10)
    ax_tx.set_aspect("equal")
    ax_tx.set_xticks([])
    ax_tx.set_yticks([])

plt.suptitle("XE2 — Segmentation Boundaries: xeNuc vs xeMultimodal", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


**Observations:**

- **xeNuc** captures only the nucleus — masks are small and tightly packed. The many transcripts
  lying outside the nuclear boundary will be **unassigned**.
- **xeMultimodal** follows the cell membrane, yielding larger masks that capture cytoplasmic
  transcripts. Compare the contours to the max-projection morphology (DAPI, 18S, interior): they extend well beyond the
  bright nuclear signal where membrane markers are visible.


In [ ]:
rows_xe2 = []
for method, info in XE2_METHODS.items():
    gdf = sdata_xe2[info["boundaries"]]
    areas = gdf.geometry.area
    rows_xe2.append({
        "Method": method,
        "Masks": len(gdf),
        "Median area (µm²)": f"{areas.median():.1f}",
        "Mean area (µm²)": f"{areas.mean():.1f}",
        "95th pctl (µm²)": f"{areas.quantile(0.95):.1f}",
    })

print(pd.DataFrame(rows_xe2).to_string(index=False))

area_frames_xe2 = []
for method, info in XE2_METHODS.items():
    gdf = sdata_xe2[info["boundaries"]]
    a = gdf.geometry.area
    if len(a) > 10_000:
        a = a.sample(10_000, random_state=42)
    area_frames_xe2.append(pd.DataFrame({"Method": method, "Area (µm²)": a.values}))

area_df_xe2 = pd.concat(area_frames_xe2, ignore_index=True)
area_df_xe2["Method"] = pd.Categorical(
    area_df_xe2["Method"],
    categories=list(XE2_METHODS.keys()),
    ordered=True,
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(
    data=area_df_xe2, x="Method", y="Area (µm²)", ax=ax,
    inner="quartile", cut=0, scale="width",
)
ax.set_ylim(0, area_df_xe2["Area (µm²)"].quantile(0.99))
ax.set_title("XE2 — Cell Mask Area Distribution")
plt.tight_layout()
plt.show()


In [ ]:
gene_panel_xe2 = set(tables_xe2["xeMultimodal"].var_names)
print("Counting gene transcripts for XE2 (QV > 20) ...")
txpts_xe2_raw = sdata_xe2["transcripts"]
txpts_xe2_hq = txpts_xe2_raw[txpts_xe2_raw["qv"] > 20].compute()
txpts_xe2_genes = txpts_xe2_hq[txpts_xe2_hq["feature_name"].isin(gene_panel_xe2)]
total_xe2_gene_txpts = len(txpts_xe2_genes)
print(f"Total gene transcripts (QV > 20): {total_xe2_gene_txpts:,}\n")

assign_xe2 = []
for method, adata in tables_xe2.items():
    assigned = int(adata.obs["total_counts"].sum())
    pct = 100 * assigned / total_xe2_gene_txpts
    assign_xe2.append({"Method": method, "Cells": adata.n_obs,
                        "Assigned": assigned, "% assigned": pct})
assign_xe2_df = pd.DataFrame(assign_xe2)
print(assign_xe2_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(
    assign_xe2_df["Method"], assign_xe2_df["% assigned"],
    color=["#e74c3c", "#27ae60"],
)
for bar, pct in zip(bars, assign_xe2_df["% assigned"]):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{pct:.1f}%", ha="center", fontsize=10)
axes[0].set_ylabel("% of gene transcripts assigned")
axes[0].set_title("XE2 — Transcript Assignment Efficiency")
axes[0].set_ylim(0, 60)

qc_frames_xe2 = []
for method, adata in tables_xe2.items():
    sub = adata.obs[["total_counts"]].copy()
    if len(sub) > 5000:
        sub = sub.sample(5000, random_state=42)
    sub["Method"] = method
    qc_frames_xe2.append(sub)
qc_xe2 = pd.concat(qc_frames_xe2, ignore_index=True)
qc_xe2["Method"] = pd.Categorical(qc_xe2["Method"], categories=list(XE2_METHODS.keys()), ordered=True)

sns.violinplot(data=qc_xe2, x="Method", y="total_counts", ax=axes[1],
               inner="quartile", cut=0, scale="width")
axes[1].set_title("XE2 — Transcripts per Cell")
axes[1].set_ylabel("total_counts")
axes[1].set_ylim(0, qc_xe2["total_counts"].quantile(0.99))
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, (method, adata) in zip(axes, tables_xe2.items()):
    sc.pl.umap(
        adata, color="celltype_broad", ax=ax, show=False,
        title=f"{method} ({adata.n_obs:,} cells)", legend_loc="none", size=10,
    )

handles = [
    mpatches.Patch(color=CELLTYPE_PALETTE[ct], label=ct)
    for ct in CELLTYPE_ORDER
    if any(ct in tables_xe2[m].obs["celltype_broad"].cat.categories for m in tables_xe2)
]
fig.legend(handles=handles, loc="lower center", ncol=len(handles),
           fontsize=9, bbox_to_anchor=(0.5, -0.06))
plt.suptitle("XE2 — UMAP by Cell Type", fontsize=14)
plt.tight_layout()
plt.show()

xe2_method_order = list(XE2_METHODS.keys())
prop_xe2 = []
for method, adata in tables_xe2.items():
    counts = adata.obs["celltype_broad"].value_counts(normalize=True)
    for ct in CELLTYPE_ORDER:
        prop_xe2.append({"Method": method, "Cell Type": ct, "Proportion": counts.get(ct, 0)})

prop_xe2_df = pd.DataFrame(prop_xe2)
pivot_xe2 = prop_xe2_df.pivot(index="Method", columns="Cell Type", values="Proportion")
pivot_xe2 = pivot_xe2[[ct for ct in CELLTYPE_ORDER if ct in pivot_xe2.columns]]
pivot_xe2 = pivot_xe2.reindex(xe2_method_order)

fig, ax = plt.subplots(figsize=(8, 5))
pivot_xe2.plot(
    kind="bar", stacked=True, ax=ax,
    color=[CELLTYPE_PALETTE[ct] for ct in pivot_xe2.columns],
    width=0.7,
)
ax.set_ylabel("Proportion of cells")
ax.set_title("XE2 — Cell Type Proportions by Segmentation Method")
ax.legend(title="Cell Type", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Cell counts for key cell types across XE2 methods
ct_focus_xe2 = ["Microglia", "Endothelial", "VLMC", "Oligo"]
focus_xe2 = []
for method, adata in tables_xe2.items():
    vc = adata.obs["celltype_broad"].value_counts()
    for ct in ct_focus_xe2:
        focus_xe2.append({"Method": method, "Cell Type": ct, "Count": vc.get(ct, 0)})

focus_xe2_df = pd.DataFrame(focus_xe2)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=focus_xe2_df, x="Cell Type", y="Count", hue="Method", ax=ax)
ax.set_title("XE2 — Cell Counts for Key Non-Neuronal Types")
ax.legend(title="Method")
plt.tight_layout()
plt.show()


### XE2 Multimodal Segmentation — Key Findings

1. **More cells and more transcripts with multimodal staining.**
   xeMultimodal captures substantially more cells than xeNuc because it can delineate
   cell bodies beyond the nucleus using membrane markers. This directly translates to
   higher transcript assignment efficiency — fewer transcripts are wasted in the
   extracellular space.

2. **Cell type composition shifts with segmentation method.**
   xeNuc over-represents cell types with large, bright nuclei (e.g. Astrocytes) relative to
   their true abundance, because cells with faint or small nuclei may be missed or merged.
   xeMultimodal recovers a more balanced composition, especially for small-bodied cell types
   like microglia and endothelial cells that are harder to detect from nuclei alone.

3. **Practical implications.**
   When multimodal stain kits are available, this is generally preferred - even with the imperfect performance of this kit in brain tissue as it still captures the most complete picture of the tissue. However, if only DAPI staining is
   available (as in XE1), a nuclear expansion approach is necessary — and the choice of
   expansion distance becomes a critical parameter that affects all downstream results. Ideally, a smarter approach than blindly expanding would be taken, which takes into consideration the spatial clustering of transcripts or even the gene identity of these transcripts to update assignments (see section 11 below).


## 10. Conclusions

The key takeaways from this analysis are:

1. **Nuclear-only segmentation (xeNuc)** is the most conservative, but captures only ~25 % of transcripts. It is the safest choice when cell-type specificity in very dense tissue is paramount but will result in a large amount of information loss and shouldn't be used in most situations.

2. **Nuclear expansion distance can have a dramatic effect on your resultant cell x gene matrix.** Expanding by 5 µm captures ~65 % of transcripts, while 10 µm captures ~83 %. However, larger expansion increases misallocation — especially for anatomically adjacent cell types like VLMCs and endothelial cells.

3. **Tightly packed cells such as VLMCs can be sensitive indicators of misallocation.** They disappear from the data at >= 5 µm expansion because endothelial transcripts overwhelm their nuclear signal.

4. **Clustering quality metrics** (silhouette scores, DEG fold changes) can help guide segmentation choices. More conservative methods tend to produce more distinct clusters, at the cost of lower transcript capture.

5. **Simple approaches to segmentation will result in non-optimal data outputs.** Where possible, use all the available information that you have gathered from your tissue to improve your segmentation using the latest image-based segmentation models (CellposeSAM). Don't assume that this has been done for you by the output straight from your instrument of choice, these often use outdated segmentation methods, even if they have access to good images from the machine. You should always consider resegmenting your data yourself.


## 11. Extra reading: Transcript-based segmentation

All methods discussed above define cell boundaries based on **images** (DAPI, protein stains, RNA probes). A supplement to this is to use the **spatial distribution of transcripts themselves** to refine or define cell boundaries.

### Some popular transcript-based segmentation tools
- **[Baysor](https://github.com/kharchenkolab/Baysor)** — A probabilistic model that uses transcript spatial patterns and (optionally) prior nuclear segmentation to jointly assign transcripts and infer cell boundaries. Particularly powerful when morphology staining is poor or unavailable.

- **[ProSeg](https://github.com/dcjones/proseg)** — Optimises cell segmentation by reassigning transcripts that were previously assigned based on image data, based on gene expression similarity between neighbouring cells. Can recover misallocated transcripts without requiring additional staining.


### Why transcript-based methods matter

Image-based segmentation relies on **morphological proxies** for cell boundaries. These proxies can fail when:

- Cells have unusual morphology (e.g. pyramidal neurons, ramified cells with long processes)
- Staining quality is uneven or tissue preservation varies
- Closely packed cells share similar morphological features

Transcript-based methods address this by leveraging **gene expression patterns** directly — if transcripts in a region cluster together in space or have expression profiles consistent with different cell types, they likely belong to different cells. This information is complementary to image-based approaches and can significantly improve segmentation quality.

These methods represent the cutting edge of spatial transcriptomics analysis and are an active area of development.
